# Comprehensive Data Exploration: Algerian Bac Educational Videos

This notebook provides an in-depth exploratory data analysis of collected YouTube video data for Algerian Baccalaureate educational content.

## Objectives
1. **Raw Data Exploration** - Understand the structure and quality of raw video metadata
2. **Channel Analysis** - Analyze channel-level statistics and coverage
3. **Transcript Analysis** - Examine transcript availability and characteristics
4. **Feature Analysis** - Explore engineered features and their distributions
5. **Cross-Dataset Analysis** - Understand relationships between data sources
6. **Data Quality Report** - Provide actionable recommendations for data cleaning

---

## Data Sources

| File | Description |
|------|-------------|
| `videos_bac_only.csv` | Raw video metadata (10K+ videos) |
| `channel_statistics.csv` | Channel-level statistics (36 channels) |
| `transcripts_merged.csv` | Collected video transcripts |
| `videos_engineered.csv` | Engineered features for modeling |

---

# Part 1: Setup & Configuration

In [ ]:
# Core imports
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed
SEED = 42
np.random.seed(SEED)

print("✅ Core imports complete")

In [ ]:
# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 100)

# Visualization settings
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palette for consistent visualizations
COLORS = {
    'primary': '#3498db',
    'secondary': '#2ecc71',
    'warning': '#f39c12',
    'danger': '#e74c3c',
    'info': '#9b59b6',
    'dark': '#34495e',
    'success': '#2ca02c',
}

print("✅ Display settings configured")

In [ ]:
# File paths
DATA_DIR = project_root / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = project_root / 'reports' / 'notebook1'

# Create reports directory if it doesn't exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Data file paths
FILES = {
    'videos_raw': PROCESSED_DIR / 'videos_bac_only.csv',
    'channels': RAW_DIR / 'channel_statistics.csv',
    'transcripts': PROCESSED_DIR / 'transcripts_merged.csv',
    'videos_engineered': PROCESSED_DIR / 'videos_engineered.csv'
}

# Verify files exist
print("📁 Data Files Status:")
for name, path in FILES.items():
    status = "✅" if path.exists() else "❌"
    print(f"  {status} {name}: {path.name}")

print(f"\n📂 Reports will be saved to: {REPORTS_DIR}")

In [ ]:
# Helper functions for EDA

def describe_df(df, name="DataFrame"):
    """Print comprehensive DataFrame summary."""
    print(f"\n{'='*60}")
    print(f"📊 {name} Summary")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Duplicates: {df.duplicated().sum():,} rows")
    
    # Data types breakdown
    dtypes = df.dtypes.value_counts()
    print(f"\nData Types:")
    for dtype, count in dtypes.items():
        print(f"  - {dtype}: {count} columns")
    
    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
    missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)
    
    if len(missing_df) > 0:
        print(f"\n⚠️ Missing Values ({len(missing_df)} columns):")
        for col, row in missing_df.head(10).iterrows():
            print(f"  - {col}: {row['count']:,} ({row['percent']:.1f}%)")
        if len(missing_df) > 10:
            print(f"  ... and {len(missing_df) - 10} more columns with missing values")
    else:
        print("\n✅ No missing values")

def plot_distribution(df, column, title=None, log_scale=False, bins=50, figsize=(14, 5)):
    """Plot distribution with histogram and box plot."""
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    data = df[column].dropna()
    title = title or column
    
    # Histogram
    axes[0].hist(data, bins=bins, edgecolor='black', alpha=0.7, color=COLORS['primary'])
    axes[0].set_xlabel(column)
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'{title} - Distribution')
    axes[0].axvline(data.mean(), color=COLORS['danger'], linestyle='--', label=f'Mean: {data.mean():.2f}')
    axes[0].axvline(data.median(), color=COLORS['secondary'], linestyle='--', label=f'Median: {data.median():.2f}')
    axes[0].legend()
    
    # Log-scale histogram (if requested or if data is skewed)
    if log_scale or data.skew() > 2:
        log_data = np.log1p(data)
        axes[1].hist(log_data, bins=bins, edgecolor='black', alpha=0.7, color=COLORS['secondary'])
        axes[1].set_xlabel(f'log(1 + {column})')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title(f'{title} - Log Scale')
    else:
        sns.kdeplot(data=data, ax=axes[1], color=COLORS['secondary'], fill=True)
        axes[1].set_xlabel(column)
        axes[1].set_ylabel('Density')
        axes[1].set_title(f'{title} - KDE')
    
    # Box plot
    axes[2].boxplot(data, vert=True)
    axes[2].set_ylabel(column)
    axes[2].set_title(f'{title} - Box Plot')
    
    plt.tight_layout()
    return fig

def save_figure(fig, name):
    """Save figure to reports directory."""
    filepath = REPORTS_DIR / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"💾 Saved: {filepath.name}")

print("✅ Helper functions defined")

---

# Part 2: Raw Data Exploration (`videos_bac_only.csv`)

## 2.1 Data Overview

In [ ]:
# Load raw video data
df_raw = pd.read_csv(FILES['videos_raw'])

# Convert date columns
if 'publish_date' in df_raw.columns:
    df_raw['publish_date'] = pd.to_datetime(df_raw['publish_date'])
if 'snapshot_date' in df_raw.columns:
    df_raw['snapshot_date'] = pd.to_datetime(df_raw['snapshot_date'])

# Overview
describe_df(df_raw, "Raw Videos (videos_bac_only.csv)")

In [ ]:
# Column information
print("\n📋 Column Details:")
print("-" * 80)

for col in df_raw.columns:
    dtype = df_raw[col].dtype
    nunique = df_raw[col].nunique()
    null_count = df_raw[col].isnull().sum()
    
    # Sample value
    sample = df_raw[col].dropna().iloc[0] if null_count < len(df_raw) else "N/A"
    if isinstance(sample, str) and len(sample) > 40:
        sample = sample[:40] + "..."
    
    print(f"{col:30s} | {str(dtype):10s} | Unique: {nunique:6,} | Null: {null_count:5,} | Sample: {sample}")

In [ ]:
# First 5 rows
print("\n📝 First 5 rows:")
df_raw.head()

## 2.2 Missing Values Analysis

In [ ]:
# Missing values detailed analysis
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values,
    'Non-Null Count': len(df_raw) - missing.values
}).sort_values('Missing Count', ascending=False)

print("\n📊 Missing Values Summary:")
print(f"Total columns: {len(df_raw.columns)}")
print(f"Columns with missing values: {(missing > 0).sum()}")
print(f"\nColumns with missing values (sorted by count):")
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(14, 8))

# Create missing matrix
missing_matrix = df_raw.isnull().astype(int)

# Only show columns with missing values
cols_with_missing = missing_matrix.columns[missing_matrix.sum() > 0]

if len(cols_with_missing) > 0:
    # Sample rows for visualization (to avoid memory issues)
    sample_size = min(500, len(df_raw))
    sample_indices = np.random.choice(len(df_raw), sample_size, replace=False)
    
    sns.heatmap(
        missing_matrix.iloc[sample_indices][cols_with_missing],
        cbar=True,
        yticklabels=False,
        cmap='YlOrRd',
        ax=ax
    )
    ax.set_title('Missing Values Heatmap (Yellow = Missing, Sample of 500 rows)')
    ax.set_xlabel('Columns')
    ax.set_ylabel('Rows (sampled)')
    plt.xticks(rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'No missing values in dataset', ha='center', va='center', fontsize=14)
    ax.set_title('Missing Values Heatmap')

plt.tight_layout()
save_figure(fig, 'missing_values_heatmap_raw')
plt.show()

## 2.3 Numerical Features

In [ ]:
# Identify numerical columns
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()

# Key numerical columns to analyze
key_numeric = ['view_count', 'like_count', 'comment_count', 'duration_sec']
key_numeric = [col for col in key_numeric if col in numeric_cols]

print(f"\n📊 Numerical Columns ({len(numeric_cols)} total):")
print(numeric_cols)
print(f"\n🎯 Key metrics to analyze: {key_numeric}")

In [ ]:
# Descriptive statistics for key numerical columns
if key_numeric:
    stats_df = df_raw[key_numeric].describe().T
    stats_df['skew'] = df_raw[key_numeric].skew()
    stats_df['kurtosis'] = df_raw[key_numeric].kurtosis()
    stats_df['null_count'] = df_raw[key_numeric].isnull().sum()
    
    print("\n📈 Key Numerical Statistics:")
    display(stats_df.round(2))
    
    # Interpretation
    print("\n🔍 Observations:")
    for col in key_numeric:
        skew = stats_df.loc[col, 'skew']
        if abs(skew) > 2:
            print(f"  ⚠️ {col}: Highly skewed ({skew:.2f}) - consider log transformation")
        elif abs(skew) > 1:
            print(f"  ⚡ {col}: Moderately skewed ({skew:.2f})")
        else:
            print(f"  ✅ {col}: Approximately symmetric ({skew:.2f})")

In [ ]:
# Distribution plots for key numerical features
for col in key_numeric:
    fig = plot_distribution(df_raw, col, title=col.replace('_', ' ').title())
    save_figure(fig, f'dist_{col}')
    plt.show()

In [ ]:
# Outlier detection using IQR method
print("\n🔍 Outlier Detection (IQR Method):")
print("-" * 60)

outlier_summary = []

for col in key_numeric:
    data = df_raw[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    outlier_pct = len(outliers) / len(data) * 100
    
    outlier_summary.append({
        'Column': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outliers': len(outliers),
        'Outlier %': outlier_pct
    })
    
    print(f"{col}:")
    print(f"  Bounds: [{lower_bound:,.0f}, {upper_bound:,.0f}]")
    print(f"  Outliers: {len(outliers):,} ({outlier_pct:.1f}%)")

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

## 2.4 Categorical Features

In [ ]:
# Identify categorical columns
categorical_cols = df_raw.select_dtypes(include=['object']).columns.tolist()

# Remove text columns (title, description) from categorical analysis
text_cols = ['title', 'description', 'tags', 'thumbnail_url']
categorical_cols = [col for col in categorical_cols if col not in text_cols]

print(f"\n📋 Categorical Columns ({len(categorical_cols)}):")
for col in categorical_cols:
    nunique = df_raw[col].nunique()
    print(f"  - {col}: {nunique} unique values")

In [ ]:
# Value counts for key categorical columns
key_categorical = ['subject', 'filter_category', 'filter_confidence']
key_categorical = [col for col in key_categorical if col in df_raw.columns]

for col in key_categorical:
    print(f"\n📊 {col.replace('_', ' ').title()} Distribution:")
    print("-" * 40)
    vc = df_raw[col].value_counts()
    vc_pct = df_raw[col].value_counts(normalize=True) * 100
    
    combined = pd.DataFrame({'Count': vc, 'Percent': vc_pct.round(2)})
    print(combined)

In [ ]:
# Bar charts for categorical features
if 'subject' in df_raw.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    subject_counts = df_raw['subject'].value_counts()
    colors = sns.color_palette('husl', len(subject_counts))
    
    bars = ax.barh(range(len(subject_counts)), subject_counts.values, color=colors)
    ax.set_yticks(range(len(subject_counts)))
    ax.set_yticklabels(subject_counts.index)
    ax.set_xlabel('Number of Videos')
    ax.set_title('Videos by Subject')
    
    # Add value labels
    for i, (count, pct) in enumerate(zip(subject_counts.values, (subject_counts / subject_counts.sum() * 100))):
        ax.text(count + 50, i, f'{count:,} ({pct:.1f}%)', va='center', fontsize=10)
    
    plt.tight_layout()
    save_figure(fig, 'subject_distribution')
    plt.show()

## 2.5 Text Features

In [ ]:
# Analyze title and description lengths
df_raw['title_length'] = df_raw['title'].fillna('').str.len()
df_raw['title_word_count'] = df_raw['title'].fillna('').str.split().str.len()
df_raw['description_length'] = df_raw['description'].fillna('').str.len()

print("\n📝 Text Feature Statistics:")
text_stats = df_raw[['title_length', 'title_word_count', 'description_length']].describe()
text_stats

In [ ]:
# Title length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw['title_length'], bins=50, edgecolor='black', alpha=0.7, color=COLORS['primary'])
axes[0].axvline(df_raw['title_length'].mean(), color=COLORS['danger'], linestyle='--', label=f"Mean: {df_raw['title_length'].mean():.0f}")
axes[0].set_xlabel('Title Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Title Length Distribution')
axes[0].legend()

axes[1].hist(df_raw['title_word_count'], bins=30, edgecolor='black', alpha=0.7, color=COLORS['secondary'])
axes[1].axvline(df_raw['title_word_count'].mean(), color=COLORS['danger'], linestyle='--', label=f"Mean: {df_raw['title_word_count'].mean():.0f}")
axes[1].set_xlabel('Title Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Title Word Count Distribution')
axes[1].legend()

plt.tight_layout()
save_figure(fig, 'text_length_distribution')
plt.show()

In [ ]:
# Sample titles (showing variety)
print("\n📝 Sample Titles (first 10):")
for i, title in enumerate(df_raw['title'].head(10), 1):
    print(f"  {i}. {title[:80]}{'...' if len(title) > 80 else ''}")

## 2.6 Temporal Analysis

In [ ]:
# Date range
if 'publish_date' in df_raw.columns:
    print("\n📅 Temporal Statistics:")
    print(f"  Date range: {df_raw['publish_date'].min()} to {df_raw['publish_date'].max()}")
    print(f"  Span: {(df_raw['publish_date'].max() - df_raw['publish_date'].min()).days:,} days")
    
    # Extract temporal features
    df_raw['publish_year'] = df_raw['publish_date'].dt.year
    df_raw['publish_month'] = df_raw['publish_date'].dt.month
    df_raw['publish_day_of_week'] = df_raw['publish_date'].dt.day_name()
    df_raw['publish_hour'] = df_raw['publish_date'].dt.hour

In [ ]:
# Videos over time
if 'publish_date' in df_raw.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Videos per year
    year_counts = df_raw['publish_year'].value_counts().sort_index()
    axes[0, 0].bar(year_counts.index, year_counts.values, color=COLORS['primary'], edgecolor='black')
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel('Number of Videos')
    axes[0, 0].set_title('Videos Published per Year')
    
    # Videos per month
    month_counts = df_raw['publish_month'].value_counts().sort_index()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[0, 1].bar(range(1, 13), [month_counts.get(i, 0) for i in range(1, 13)], color=COLORS['secondary'], edgecolor='black')
    axes[0, 1].set_xticks(range(1, 13))
    axes[0, 1].set_xticklabels(month_names, rotation=45)
    axes[0, 1].set_xlabel('Month')
    axes[0, 1].set_ylabel('Number of Videos')
    axes[0, 1].set_title('Videos Published per Month')
    
    # Videos per day of week
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts = df_raw['publish_day_of_week'].value_counts().reindex(day_order)
    axes[1, 0].bar(range(7), day_counts.values, color=COLORS['info'], edgecolor='black')
    axes[1, 0].set_xticks(range(7))
    axes[1, 0].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    axes[1, 0].set_xlabel('Day of Week')
    axes[1, 0].set_ylabel('Number of Videos')
    axes[1, 0].set_title('Videos Published per Day of Week')
    
    # Videos per hour
    hour_counts = df_raw['publish_hour'].value_counts().sort_index()
    axes[1, 1].bar(range(24), [hour_counts.get(i, 0) for i in range(24)], color=COLORS['warning'], edgecolor='black')
    axes[1, 1].set_xlabel('Hour of Day')
    axes[1, 1].set_ylabel('Number of Videos')
    axes[1, 1].set_title('Videos Published per Hour')
    axes[1, 1].axvspan(17, 21, alpha=0.3, color='green', label='Evening (5-9 PM)')
    axes[1, 1].legend()
    
    plt.tight_layout()
    save_figure(fig, 'temporal_analysis')
    plt.show()

In [ ]:
# Videos over time (monthly trend)
if 'publish_date' in df_raw.columns:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    monthly_counts = df_raw.set_index('publish_date').resample('ME').size()
    
    ax.plot(monthly_counts.index, monthly_counts.values, marker='o', linewidth=2, markersize=4, color=COLORS['primary'])
    ax.fill_between(monthly_counts.index, monthly_counts.values, alpha=0.3, color=COLORS['primary'])
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Videos')
    ax.set_title('Monthly Video Publishing Trend')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    save_figure(fig, 'monthly_trend')
    plt.show()

---

# Part 2 Summary

**Key findings from raw data exploration will be summarized here after running the notebook.**

---

# Part 3: Channel Statistics Analysis (`channel_statistics.csv`)

## 3.1 Channel Overview

In [ ]:
# Load channel statistics
df_channels = pd.read_csv(FILES['channels'])

# Convert date columns (use format='ISO8601' to handle mixed formats)
if 'created_at' in df_channels.columns:
    df_channels['created_at'] = pd.to_datetime(df_channels['created_at'], format='ISO8601')
if 'snapshot_date' in df_channels.columns:
    df_channels['snapshot_date'] = pd.to_datetime(df_channels['snapshot_date'], format='ISO8601')

# Overview
describe_df(df_channels, "Channel Statistics (channel_statistics.csv)")

In [ ]:
# Channel details
print("\n📋 All Channels:")
print("-" * 100)

display_cols = ['title', 'subscriber_count', 'video_count', 'view_count']
display_cols = [c for c in display_cols if c in df_channels.columns]

if display_cols:
    channel_summary = df_channels[display_cols].copy()
    channel_summary = channel_summary.sort_values('subscriber_count', ascending=False)
    display(channel_summary)

In [ ]:
# Channel statistics summary
print("\n📊 Channel Statistics Summary:")
print("-" * 60)

if 'subscriber_count' in df_channels.columns:
    print(f"Total channels: {len(df_channels)}")
    print(f"Total subscribers: {df_channels['subscriber_count'].sum():,.0f}")
    print(f"Total views: {df_channels['view_count'].sum():,.0f}")
    print(f"Total videos: {df_channels['video_count'].sum():,.0f}")
    print(f"\nSubscriber range: {df_channels['subscriber_count'].min():,.0f} - {df_channels['subscriber_count'].max():,.0f}")
    print(f"Median subscribers: {df_channels['subscriber_count'].median():,.0f}")

## 3.2 Channel Performance Metrics

In [ ]:
# Subscriber distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_channels['subscriber_count'], bins=15, edgecolor='black', alpha=0.7, color=COLORS['primary'])
axes[0].set_xlabel('Subscriber Count')
axes[0].set_ylabel('Number of Channels')
axes[0].set_title('Subscriber Count Distribution')
axes[0].axvline(df_channels['subscriber_count'].median(), color=COLORS['danger'], linestyle='--', 
                label=f"Median: {df_channels['subscriber_count'].median():,.0f}")
axes[0].legend()

# Top 10 channels by subscribers
top_channels = df_channels.nlargest(10, 'subscriber_count')
axes[1].barh(range(len(top_channels)), top_channels['subscriber_count'].values, color=COLORS['secondary'])
axes[1].set_yticks(range(len(top_channels)))
axes[1].set_yticklabels(top_channels['title'].str[:30])
axes[1].set_xlabel('Subscribers')
axes[1].set_title('Top 10 Channels by Subscribers')
axes[1].invert_yaxis()

plt.tight_layout()
save_figure(fig, 'channel_subscribers')
plt.show()

In [ ]:
# Channel age analysis
if 'created_at' in df_channels.columns:
    from datetime import datetime
    
    df_channels['channel_age_days'] = (pd.Timestamp.now(tz='UTC') - df_channels['created_at']).dt.days
    df_channels['channel_age_years'] = df_channels['channel_age_days'] / 365.25
    
    print("\n📅 Channel Age Statistics:")
    print(f"  Oldest channel: {df_channels['channel_age_years'].max():.1f} years")
    print(f"  Newest channel: {df_channels['channel_age_years'].min():.1f} years")
    print(f"  Average age: {df_channels['channel_age_years'].mean():.1f} years")

## 3.3 Channel Coverage Check

In [ ]:
# Check how many videos we have per channel
videos_per_channel = df_raw.groupby('channel_id').size().reset_index(name='videos_in_dataset')

# Merge with channel stats
channel_coverage = df_channels.merge(
    videos_per_channel, 
    left_on='channel_id', 
    right_on='channel_id', 
    how='left'
)
channel_coverage['videos_in_dataset'] = channel_coverage['videos_in_dataset'].fillna(0).astype(int)
channel_coverage['coverage_pct'] = (channel_coverage['videos_in_dataset'] / channel_coverage['video_count'] * 100).round(2)

print("\n📊 Video Coverage by Channel:")
print("-" * 80)
coverage_display = channel_coverage[['title', 'video_count', 'videos_in_dataset', 'coverage_pct']].copy()
coverage_display.columns = ['Channel', 'Total Videos', 'In Dataset', 'Coverage %']
coverage_display = coverage_display.sort_values('In Dataset', ascending=False)
display(coverage_display)

print(f"\n✅ Total videos in raw data: {len(df_raw):,}")
print(f"✅ Channels with videos in dataset: {(channel_coverage['videos_in_dataset'] > 0).sum()}/{len(df_channels)}")

In [ ]:
# Coverage visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Sort by videos in dataset
sorted_coverage = channel_coverage.sort_values('videos_in_dataset', ascending=True)

y_pos = range(len(sorted_coverage))
ax.barh(y_pos, sorted_coverage['videos_in_dataset'], color=COLORS['primary'], label='In Dataset')
ax.barh(y_pos, sorted_coverage['video_count'] - sorted_coverage['videos_in_dataset'], 
        left=sorted_coverage['videos_in_dataset'], color=COLORS['warning'], alpha=0.5, label='Not in Dataset')

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_coverage['title'].str[:25])
ax.set_xlabel('Number of Videos')
ax.set_title('Video Coverage by Channel')
ax.legend()

plt.tight_layout()
save_figure(fig, 'channel_video_coverage')
plt.show()

---

## Part 4: Transcript Analysis

Analyzing the transcript data to understand coverage, quality, and text characteristics.

In [ ]:
# Load transcripts data
transcripts_path = project_root / 'data' / 'processed' / 'transcripts_merged.csv'
df_transcripts = pd.read_csv(transcripts_path)

print("=" * 80)
print("📝 TRANSCRIPT DATA OVERVIEW")
print("=" * 80)

describe_df(df_transcripts)

In [ ]:
# Display column info
print("\n📋 Column Information:")
print("-" * 80)
for col in df_transcripts.columns:
    dtype = df_transcripts[col].dtype
    non_null = df_transcripts[col].notna().sum()
    null_pct = df_transcripts[col].isna().sum() / len(df_transcripts) * 100
    print(f"  {col:30} | {str(dtype):15} | {non_null:,} non-null | {null_pct:.2f}% missing")

print("\n📄 Sample Rows:")
display(df_transcripts.head(10))

In [ ]:
# 4.1 Transcript Availability Analysis
print("\n" + "=" * 80)
print("📊 TRANSCRIPT AVAILABILITY ANALYSIS")
print("=" * 80)

# Check for transcript_available column or derive from transcript_text
if 'transcript_available' in df_transcripts.columns:
    availability = df_transcripts['transcript_available'].value_counts()
else:
    # Derive from transcript_text - available if not null and not empty
    df_transcripts['transcript_available'] = (
        df_transcripts['transcript_text'].notna() & 
        (df_transcripts['transcript_text'].str.strip() != '')
    )
    availability = df_transcripts['transcript_available'].value_counts()

success_count = availability.get(True, 0)
fail_count = availability.get(False, 0)
total = len(df_transcripts)

print(f"\n✅ Successful transcripts: {success_count:,} ({success_count/total*100:.1f}%)")
print(f"❌ Failed transcripts: {fail_count:,} ({fail_count/total*100:.1f}%)")
print(f"📊 Total records: {total:,}")

In [ ]:
# Define color palette to fix KeyError

# Visualize transcript availability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors_pie = [COLORS['success'], COLORS['danger']]
labels = ['Available', 'Not Available']
sizes = [success_count, fail_count]
explode = (0.05, 0)

axes[0].pie(sizes, explode=explode, labels=labels, colors=colors_pie, autopct='%1.1f%%',
            shadow=False, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Transcript Availability Rate', fontsize=14, fontweight='bold')

# Bar chart comparison with video count
bar_data = pd.DataFrame({
    'Category': ['Total Videos', 'With Transcript', 'Without Transcript'],
    'Count': [len(df_raw), success_count, fail_count],
    'Color': [COLORS['primary'], COLORS['success'], COLORS['danger']]
})

bars = axes[1].bar(bar_data['Category'], bar_data['Count'], color=bar_data['Color'])
axes[1].set_ylabel('Number of Videos')
axes[1].set_title('Transcript Coverage', fontsize=14, fontweight='bold')

# Add value labels
for bar in bars:
    height = bar.get_height()
    axes[1].annotate(f'{height:,}',
                     xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3), textcoords="offset points",
                     ha='center', va='bottom', fontsize=11)

plt.tight_layout()
save_figure(fig, 'transcript_availability')
plt.show()

In [ ]:
# Refined Failure Reason Analysis
print("\n" + "=" * 80)
print("❌ FAILURE REASON ANALYSIS (REFINED)")
print("=" * 80)

if 'failure_reason' in df_transcripts.columns:
    # 1. Extract failure reasons
    failures = df_transcripts[df_transcripts['transcript_available'] == False]['failure_reason'].astype(str)
    
    # 2. Define cleaning function
    def clean_failure_reason(reason):
        if "Sign in to confirm" in reason:
            return "Bot Detection (Sign in required)"
        if "Video unavailable" in reason:
            return "Video Unavailable"
        if "Private video" in reason:
            return "Private Video"
        if "Members-only" in reason:
            return "Members-only Video"
        if "No transcript found" in reason:
            return "No Transcript Available"
        if "Transcripts disabled" in reason:
            return "Transcripts Disabled"
        if "Subtitles are disabled" in reason:
            return "Subtitles Disabled"
        return "Other/Unknown Error"

    # 3. Apply cleaning
    cleaned_failures = failures.apply(clean_failure_reason)
    failure_counts = cleaned_failures.value_counts()
    
    # 4. Print Summary
    print("\n📋 Detailed Failure Breakdown:")
    print("-" * 60)
    total_failures = len(failures)
    for reason, count in failure_counts.items():
        pct = (count / total_failures) * 100
        print(f"  {reason:40} | {count:5,} ({pct:5.1f}%)")
        
    # 5. Visualization
    if len(failure_counts) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        # Colors: Danger for errors
        colors = [COLORS['danger']] * len(failure_counts)
        bars = ax.barh(range(len(failure_counts)), failure_counts.values, color=colors)
        ax.set_yticks(range(len(failure_counts)))
        ax.set_yticklabels(failure_counts.index)
        ax.set_xlabel('Number of Videos')
        ax.set_title('Reasons for Missing Transcripts', fontsize=14, fontweight='bold')
        ax.invert_yaxis() # Top reason at top
        
        # Add labels
        for i, v in enumerate(failure_counts.values):
            ax.text(v + 10, i, f"{v:,} ({v/total_failures:.1%})", va='center')
            
        plt.tight_layout()
        save_figure(fig, 'transcript_failure_analysis')
        plt.show()
else:
    print("No failure_reason column found.")


In [ ]:
# 4.2 Transcript Characteristics
print("\n" + "=" * 80)
print("🌍 TRANSCRIPT LANGUAGE DISTRIBUTION")
print("=" * 80)

if 'transcript_language' in df_transcripts.columns:
    # Only consider transcripts that are available
    available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True]
    
    lang_dist = available_transcripts['transcript_language'].value_counts()
    
    print("\n📋 Language Distribution (Available Transcripts):")
    print("-" * 60)
    for lang, count in lang_dist.items():
        pct = count / len(available_transcripts) * 100
        print(f"  {str(lang):20} | {count:,} ({pct:.1f}%)")
    
    # Visualize language distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors_lang = plt.cm.tab10(np.linspace(0, 1, len(lang_dist)))
    bars = ax.bar(range(len(lang_dist)), lang_dist.values, color=colors_lang)
    ax.set_xticks(range(len(lang_dist)))
    ax.set_xticklabels(lang_dist.index, rotation=45, ha='right')
    ax.set_ylabel('Number of Transcripts')
    ax.set_xlabel('Language Code')
    ax.set_title('Transcript Language Distribution', fontsize=14, fontweight='bold')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height):,}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    save_figure(fig, 'transcript_languages')
    plt.show()
else:
    print("\n⚠️ No 'transcript_language' column found")

In [ ]:
# Auto-generated vs Manual transcripts
print("\n" + "=" * 80)
print("🤖 AUTO-GENERATED vs MANUAL TRANSCRIPTS")
print("=" * 80)

if 'is_generated' in df_transcripts.columns:
    available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True]
    
    generated_dist = available_transcripts['is_generated'].value_counts()
    
    print("\n📋 Transcript Type Distribution:")
    print("-" * 60)
    for gen_type, count in generated_dist.items():
        pct = count / len(available_transcripts) * 100
        label = "Auto-generated" if gen_type else "Manual/Uploaded"
        print(f"  {label:20} | {count:,} ({pct:.1f}%)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(8, 6))
    
    labels = ['Auto-generated' if x else 'Manual/Uploaded' for x in generated_dist.index]
    colors_gen = [COLORS['warning'], COLORS['success']]
    
    ax.pie(generated_dist.values, labels=labels, colors=colors_gen, autopct='%1.1f%%',
           startangle=90, textprops={'fontsize': 12})
    ax.set_title('Auto-generated vs Manual Transcripts', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, 'transcript_auto_vs_manual')
    plt.show()
else:
    print("\n⚠️ No 'is_generated' column found")

In [ ]:
# Transcript text characteristics
print("\n" + "=" * 80)
print("📏 TRANSCRIPT TEXT CHARACTERISTICS")
print("=" * 80)

available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True].copy()

if 'transcript_text' in available_transcripts.columns and len(available_transcripts) > 0:
    # Calculate text statistics
    available_transcripts['char_count'] = available_transcripts['transcript_text'].str.len()
    available_transcripts['word_count'] = available_transcripts['transcript_text'].str.split().str.len()
    
    print("\n📊 Character Count Statistics:")
    print(available_transcripts['char_count'].describe().round(2))
    
    print("\n📊 Word Count Statistics:")
    print(available_transcripts['word_count'].describe().round(2))
    
    # Visualize distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Character count distribution
    axes[0].hist(available_transcripts['char_count'].dropna(), bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
    axes[0].axvline(available_transcripts['char_count'].median(), color=COLORS['danger'], linestyle='--', linewidth=2, label=f"Median: {available_transcripts['char_count'].median():,.0f}")
    axes[0].set_xlabel('Character Count')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Transcript Character Count Distribution', fontsize=12, fontweight='bold')
    axes[0].legend()
    
    # Word count distribution
    axes[1].hist(available_transcripts['word_count'].dropna(), bins=50, color=COLORS['secondary'], edgecolor='white', alpha=0.7)
    axes[1].axvline(available_transcripts['word_count'].median(), color=COLORS['danger'], linestyle='--', linewidth=2, label=f"Median: {available_transcripts['word_count'].median():,.0f}")
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Transcript Word Count Distribution', fontsize=12, fontweight='bold')
    axes[1].legend()
    
    plt.tight_layout()
    save_figure(fig, 'transcript_text_lengths')
    plt.show()
else:
    print("\n⚠️ No transcript_text data available for analysis")

In [ ]:
# Segment count analysis
print("\n" + "=" * 80)
print("📊 SEGMENT COUNT ANALYSIS")
print("=" * 80)

if 'segment_count' in df_transcripts.columns:
    available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True]
    
    print("\n📊 Segment Count Statistics:")
    print(available_transcripts['segment_count'].describe().round(2))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.hist(available_transcripts['segment_count'].dropna(), bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
    ax.axvline(available_transcripts['segment_count'].median(), color=COLORS['danger'], linestyle='--', linewidth=2, 
               label=f"Median: {available_transcripts['segment_count'].median():.0f}")
    ax.set_xlabel('Segment Count')
    ax.set_ylabel('Frequency')
    ax.set_title('Transcript Segment Count Distribution', fontsize=14, fontweight='bold')
    ax.legend()
    
    plt.tight_layout()
    save_figure(fig, 'transcript_segment_count')
    plt.show()
else:
    print("\n⚠️ No 'segment_count' column found")

In [ ]:
# 4.3 Transcript Quality
print("\n" + "=" * 80)
print("🔍 TRANSCRIPT QUALITY ANALYSIS")
print("=" * 80)

available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True].copy()

if 'transcript_text' in available_transcripts.columns:
    # Calculate text length for quality checks
    available_transcripts['char_count'] = available_transcripts['transcript_text'].str.len()
    available_transcripts['word_count'] = available_transcripts['transcript_text'].str.split().str.len()
    
    # Define quality thresholds
    very_short_threshold = 50  # Less than 50 words
    short_threshold = 200  # Less than 200 words
    
    very_short = available_transcripts[available_transcripts['word_count'] < very_short_threshold]
    short = available_transcripts[(available_transcripts['word_count'] >= very_short_threshold) & 
                                   (available_transcripts['word_count'] < short_threshold)]
    normal = available_transcripts[available_transcripts['word_count'] >= short_threshold]
    
    print("\n📋 Transcript Length Categories:")
    print("-" * 60)
    print(f"  Very short (<{very_short_threshold} words):  {len(very_short):,} ({len(very_short)/len(available_transcripts)*100:.1f}%)")
    print(f"  Short ({very_short_threshold}-{short_threshold} words):        {len(short):,} ({len(short)/len(available_transcripts)*100:.1f}%)")
    print(f"  Normal (>={short_threshold} words):       {len(normal):,} ({len(normal)/len(available_transcripts)*100:.1f}%)")
    
    # Visualize quality distribution
    fig, ax = plt.subplots(figsize=(8, 6))
    
    quality_data = pd.DataFrame({
        'Category': [f'Very Short\n(<{very_short_threshold} words)', 
                     f'Short\n({very_short_threshold}-{short_threshold} words)', 
                     f'Normal\n(≥{short_threshold} words)'],
        'Count': [len(very_short), len(short), len(normal)],
        'Color': [COLORS['danger'], COLORS['warning'], COLORS['success']]
    })
    
    bars = ax.bar(quality_data['Category'], quality_data['Count'], color=quality_data['Color'])
    ax.set_ylabel('Number of Transcripts')
    ax.set_title('Transcript Quality by Word Count', fontsize=14, fontweight='bold')
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height):,}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    save_figure(fig, 'transcript_quality_categories')
    plt.show()

In [ ]:
# Sample transcript text review
print("\n" + "=" * 80)
print("📝 SAMPLE TRANSCRIPT REVIEW")
print("=" * 80)

available_transcripts = df_transcripts[df_transcripts['transcript_available'] == True].copy()

if 'transcript_text' in available_transcripts.columns:
    # Sample a few transcripts (short, medium, long)
    available_transcripts['word_count'] = available_transcripts['transcript_text'].str.split().str.len()
    
    # Get samples from different length categories
    samples = []
    
    # Short transcript sample
    short_sample = available_transcripts[available_transcripts['word_count'] < 100].sample(1, random_state=SEED)
    if len(short_sample) > 0:
        samples.append(('Short', short_sample.iloc[0]))
    
    # Medium transcript sample
    medium_sample = available_transcripts[(available_transcripts['word_count'] >= 200) & 
                                          (available_transcripts['word_count'] <= 500)].sample(1, random_state=SEED)
    if len(medium_sample) > 0:
        samples.append(('Medium', medium_sample.iloc[0]))
    
    # Long transcript sample
    long_sample = available_transcripts[available_transcripts['word_count'] > 1000].sample(1, random_state=SEED)
    if len(long_sample) > 0:
        samples.append(('Long', long_sample.iloc[0]))
    
    for category, sample in samples:
        print(f"\n{'='*40}")
        print(f"📄 {category} Transcript Sample (Video: {sample['video_id']})")
        print(f"{'='*40}")
        print(f"Word count: {sample['word_count']}")
        if 'transcript_language' in sample.index:
            print(f"Language: {sample['transcript_language']}")
        print(f"\nText preview (first 500 characters):")
        print("-" * 40)
        text = str(sample['transcript_text'])[:500]
        print(text + "..." if len(str(sample['transcript_text'])) > 500 else text)

In [ ]:
# Transcript coverage by subject (join with raw data)
print("\n" + "=" * 80)
print("📊 TRANSCRIPT COVERAGE BY SUBJECT")
print("=" * 80)

# Merge transcripts with raw video data to get subject info
if 'subject' in df_raw.columns:
    transcript_subject = df_raw[['video_id', 'subject']].merge(
        df_transcripts[['video_id', 'transcript_available']], 
        on='video_id', 
        how='left'
    )
    
    # Fill NaN transcript_available with False
    transcript_subject['transcript_available'] = transcript_subject['transcript_available'].fillna(False)
    
    # Calculate coverage by subject
    subject_coverage = transcript_subject.groupby('subject').agg(
        total_videos=('video_id', 'count'),
        with_transcript=('transcript_available', 'sum')
    ).reset_index()
    subject_coverage['coverage_pct'] = subject_coverage['with_transcript'] / subject_coverage['total_videos'] * 100
    subject_coverage = subject_coverage.sort_values('total_videos', ascending=False)
    
    print("\n📋 Coverage by Subject:")
    print("-" * 80)
    display(subject_coverage)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(subject_coverage))
    width = 0.35
    
    bars1 = ax.bar([i - width/2 for i in x], subject_coverage['total_videos'], width, 
                   label='Total Videos', color=COLORS['primary'], alpha=0.7)
    bars2 = ax.bar([i + width/2 for i in x], subject_coverage['with_transcript'], width, 
                   label='With Transcript', color=COLORS['success'], alpha=0.7)
    
    ax.set_xlabel('Subject')
    ax.set_ylabel('Number of Videos')
    ax.set_title('Transcript Coverage by Subject', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(subject_coverage['subject'], rotation=45, ha='right')
    ax.legend()
    
    # Add percentage labels
    for i, (idx, row) in enumerate(subject_coverage.iterrows()):
        ax.annotate(f'{row["coverage_pct"]:.0f}%',
                   xy=(i, max(row['total_videos'], row['with_transcript']) + 50),
                   ha='center', fontsize=9, fontweight='bold', color=COLORS['info'])
    
    plt.tight_layout()
    save_figure(fig, 'transcript_coverage_by_subject')
    plt.show()
else:
    print("⚠️ No 'subject' column found in raw data")

 --- 
 # Part 5: Engineered Features Analysis
 Analyzing the engineered features dataset to understand distributions, missing values, and relationships.

In [ ]:
# Load engineered features data
engineered_path = project_root / 'data' / 'processed' / 'videos_engineered.csv'
df_engineered = pd.read_csv(engineered_path)

print("=" * 80)
print("⚙️ ENGINEERED FEATURES DATA OVERVIEW")
print("=" * 80)

describe_df(df_engineered)

In [ ]:
# 5.1 Feature Categories Overview
print("\n" + "=" * 80)
print("📋 FEATURE CATEGORIES OVERVIEW")
print("=" * 80)

# Define feature categories based on the engineering pipeline
feature_categories = {
    'Temporal': ['days_since_publish', 'publish_hour', 'publish_month', 'publish_year', 
                 'is_evening_upload', 'is_weekday', 'publish_hour_sin', 'publish_hour_cos',
                 'publish_month_sin', 'publish_month_cos', 'publish_day_of_month_sin', 
                 'publish_day_of_month_cos', 'days_since_publish_squared', 'log_days_since_publish',
                 'duration_sec'],
    'Text': ['title_length', 'description_length', 'title_word_count', 'tag_count', 'is_exam_focused',
             'subject_keyword_count', 'subject_keyword_density'],
    'Engagement_Raw': ['view_count', 'like_count', 'comment_count'],
    'Engagement_Ratios': ['like_ratio', 'comment_ratio', 'engagement_score'],
    'Channel': ['channel_video_count', 'channel_avg_views', 'channel_avg_duration', 
                'channel_age_days', 'channel_subscriber_count', 'channel_title', 'channel_video_avg_duration'],
    'Transcript_Basic': ['transcript_word_count', 'transcript_char_count', 'transcript_sentence_count',
                         'has_transcript', 'lexical_diversity', 'unique_word_count'],
    'Transcript_Readability': ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog_index', 
                                'automated_readability_index', 'speech_rate_optimal', 
                                'avg_words_per_sentence', 'speech_rate_wpm'],
    'Transcript_Pedagogical': ['question_count', 'example_count', 'explanation_count', 
                               'contrast_count', 'technical_term_count', 'contrast_density',
                               'example_density', 'explanation_density', 'question_density', 
                               'technical_term_density']
}

# Check which features exist in the dataset
print("\n📊 Feature Category Breakdown:")
print("-" * 80)

all_features = set(df_engineered.columns)
categorized_features = []

for category, features in feature_categories.items():
    existing = [f for f in features if f in all_features]
    categorized_features.extend(existing)
    print(f"\n{category} ({len(existing)} features):")
    for f in existing:
        missing_pct = df_engineered[f].isna().sum() / len(df_engineered) * 100
        print(f"  - {f}: {missing_pct:.1f}% missing")

# Uncategorized features
uncategorized = all_features - set(categorized_features) - {'video_id', 'channel_id', 'title', 
                                                            'description', 'publish_date', 'subject',
                                                            'publish_day_of_week', 'engagement_category'}
if uncategorized:
    print(f"\nUncategorized ({len(uncategorized)} features):")
    for f in sorted(uncategorized):
        print(f"  - {f}")

In [ ]:
# 5.2 Missing Values Deep Dive
print("\n" + "=" * 80)
print("🔍 MISSING VALUES DEEP DIVE")
print("=" * 80)

# Calculate missing values for all columns
missing_data = pd.DataFrame({
    'Column': df_engineered.columns,
    'Missing_Count': df_engineered.isna().sum().values,
    'Missing_Pct': (df_engineered.isna().sum() / len(df_engineered) * 100).values,
    'Dtype': df_engineered.dtypes.values
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Pct', ascending=False)

print(f"\n📋 Columns with Missing Values ({len(missing_data)} out of {len(df_engineered.columns)}):")
print("-" * 80)
display(missing_data)

In [ ]:
# Missing values heatmap for numerical features
print("\n📊 Missing Values Pattern Visualization:")

# Select numerical columns with any missing values
numerical_cols = df_engineered.select_dtypes(include=[np.number]).columns.tolist()
cols_with_missing = [col for col in numerical_cols if df_engineered[col].isna().sum() > 0]

if len(cols_with_missing) > 0:
    # Create missingness matrix (sample for performance)
    sample_size = min(1000, len(df_engineered))
    sample_df = df_engineered[cols_with_missing].sample(sample_size, random_state=SEED)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create binary missingness matrix
    missing_matrix = sample_df.isna().astype(int)
    
    sns.heatmap(missing_matrix.T, cmap='YlOrRd', cbar_kws={'label': 'Missing (1) vs Present (0)'},
                yticklabels=True, xticklabels=False, ax=ax)
    ax.set_xlabel(f'Samples (n={sample_size})')
    ax.set_ylabel('Features')
    ax.set_title('Missing Values Pattern (Features with Missing Data)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, 'engineered_missing_pattern')
    plt.show()
else:
    print("✅ No missing values in numerical features!")

In [ ]:
# Missing values bar chart
if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(12, max(6, len(missing_data) * 0.3)))
    
    # Color by severity
    colors = []
    for pct in missing_data['Missing_Pct']:
        if pct > 50:
            colors.append(COLORS['danger'])
        elif pct > 20:
            colors.append(COLORS['warning'])
        else:
            colors.append(COLORS['info'])
    
    bars = ax.barh(range(len(missing_data)), missing_data['Missing_Pct'], color=colors)
    ax.set_yticks(range(len(missing_data)))
    ax.set_yticklabels(missing_data['Column'])
    ax.set_xlabel('Missing Percentage (%)')
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.axvline(x=20, color='orange', linestyle='--', alpha=0.5, label='20% threshold')
    ax.legend()
    
    # Add percentage labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.annotate(f'{width:.1f}%',
                   xy=(width, bar.get_y() + bar.get_height() / 2),
                   xytext=(5, 0), textcoords="offset points",
                   ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    save_figure(fig, 'engineered_missing_values_bar')
    plt.show()

In [ ]:
# 5.3 Target Variable Analysis
print("\n" + "=" * 80)
print("🎯 TARGET VARIABLE ANALYSIS: engagement_score")
print("=" * 80)

if 'engagement_score' in df_engineered.columns:
    target = df_engineered['engagement_score'].dropna()
    
    print(f"\n📊 Basic Statistics:")
    print("-" * 60)
    print(f"  Count: {len(target):,}")
    print(f"  Mean: {target.mean():.4f}")
    print(f"  Median: {target.median():.4f}")
    print(f"  Std: {target.std():.4f}")
    print(f"  Min: {target.min():.4f}")
    print(f"  Max: {target.max():.4f}")
    print(f"  Skewness: {target.skew():.4f}")
    print(f"  Kurtosis: {target.kurtosis():.4f}")
    
    # Outlier detection using IQR
    Q1 = target.quantile(0.25)
    Q3 = target.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = target[(target < lower_bound) | (target > upper_bound)]
    
    print(f"\n📊 Outlier Analysis (IQR Method):")
    print("-" * 60)
    print(f"  Q1: {Q1:.4f}")
    print(f"  Q3: {Q3:.4f}")
    print(f"  IQR: {IQR:.4f}")
    print(f"  Lower bound: {lower_bound:.4f}")
    print(f"  Upper bound: {upper_bound:.4f}")
    print(f"  Outliers: {len(outliers):,} ({len(outliers)/len(target)*100:.1f}%)")
else:
    print("⚠️ No 'engagement_score' column found!")

In [ ]:
# Target variable distribution visualization
if 'engagement_score' in df_engineered.columns:
    target = df_engineered['engagement_score'].dropna()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Histogram with KDE
    axes[0, 0].hist(target, bins=50, density=True, color=COLORS['primary'], edgecolor='white', alpha=0.7)
    target.plot(kind='kde', ax=axes[0, 0], color=COLORS['danger'], linewidth=2)
    axes[0, 0].axvline(target.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {target.mean():.4f}')
    axes[0, 0].axvline(target.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {target.median():.4f}')
    axes[0, 0].set_xlabel('Engagement Score')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].set_title('Log-Transformed Engagement Score Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    
    # 2. Log-transformed distribution
    org_target = np.expm1(target)
    axes[0, 1].hist(org_target, bins=50, density=True, color=COLORS['secondary'], edgecolor='white', alpha=0.7)
    org_target.plot(kind='kde', ax=axes[0, 1], color=COLORS['danger'], linewidth=2)
    axes[0, 1].set_xlabel('Log(1 + Engagement Score)')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].set_title('Original Engagement Score', fontsize=12, fontweight='bold')
    
    # 3. Box plot
    axes[1, 0].boxplot(target, vert=True, patch_artist=True,
                       boxprops=dict(facecolor=COLORS['primary'], alpha=0.7))
    axes[1, 0].set_ylabel('Engagement Score')
    axes[1, 0].set_title('Engagement Score Box Plot', fontsize=12, fontweight='bold')
    
    # 4. Engagement category distribution (if exists)
    if 'engagement_category' in df_engineered.columns:
        cat_counts = df_engineered['engagement_category'].value_counts()
        colors_cat = [COLORS['danger'], COLORS['warning'], COLORS['info'], COLORS['success']][:len(cat_counts)]
        axes[1, 1].bar(cat_counts.index, cat_counts.values, color=colors_cat)
        axes[1, 1].set_xlabel('Engagement Category')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Engagement Category Distribution', fontsize=12, fontweight='bold')
        for i, v in enumerate(cat_counts.values):
            axes[1, 1].annotate(f'{v:,}', xy=(i, v), xytext=(0, 3), 
                               textcoords="offset points", ha='center', fontsize=10)
    else:
        axes[1, 1].text(0.5, 0.5, 'No engagement_category column', ha='center', va='center', 
                        transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].set_title('Engagement Category Distribution', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, 'engagement_score_analysis')
    plt.show()

In [ ]:
# 5.4 Feature Distributions
print("\n" + "=" * 80)
print("📊 FEATURE DISTRIBUTIONS")
print("=" * 80)

# Select key numerical features for distribution analysis
key_features = [
    'view_count', 'like_count', 'comment_count',
    'like_ratio', 'comment_ratio',
    'days_since_publish', 'title_length', 'title_word_count',
    'transcript_word_count', 'flesch_reading_ease'
]

# Filter to features that exist
existing_features = [f for f in key_features if f in df_engineered.columns]

if len(existing_features) > 0:
    n_cols = 3
    n_rows = (len(existing_features) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes.flatten()
    
    for i, feature in enumerate(existing_features):
        data = df_engineered[feature].dropna()
        
        axes[i].hist(data, bins=50, color=COLORS['primary'], edgecolor='white', alpha=0.7)
        axes[i].axvline(data.mean(), color=COLORS['danger'], linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
        axes[i].axvline(data.median(), color=COLORS['warning'], linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')
        axes[i].set_title(f'{feature} Distribution', fontsize=10, fontweight='bold')
        axes[i].legend(fontsize=8)
    
    # Hide unused subplots
    for j in range(len(existing_features), len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    save_figure(fig, 'feature_distributions_grid')
    plt.show()

In [ ]:
# 5.5 Correlation Analysis
print("\n" + "=" * 80)
print("🔗 CORRELATION ANALYSIS")
print("=" * 80)

# Select numerical features for correlation
numerical_cols = df_engineered.select_dtypes(include=[np.number]).columns.tolist()

# Remove ID columns and keep features with < 50% missing
valid_cols = [col for col in numerical_cols 
              if col not in ['video_id'] and df_engineered[col].notna().sum() / len(df_engineered) > 0.5]

# Compute correlation matrix
corr_matrix = df_engineered[valid_cols].corr()

print(f"\n📊 Correlation matrix: {len(valid_cols)} features")

In [ ]:
# Correlation heatmap
if len(valid_cols) <= 36:
    fig, ax = plt.subplots(figsize=(16, 14))
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0,
                square=True, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Correlation Coefficient'})
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, 'correlation_heatmap')
    plt.show()
else:
    print(f"⚠️ Too many features ({len(valid_cols)}) for full heatmap. Showing subset...")
    
    # Show correlation with target only
    if 'engagement_score' in valid_cols:
        target_corr = corr_matrix['engagement_score'].drop('engagement_score').sort_values(ascending=False)
        print("\n📊 Top correlations with engagement_score shown in next cell")

In [ ]:
# Top correlations with engagement_score
if 'engagement_score' in df_engineered.columns:
    target_correlations = corr_matrix['engagement_score'].drop(['engagement_score',"like_count","view_count","comment_count","like_ratio"]).sort_values(key=abs, ascending=False)
    
    print("\n🎯 Top 20 Correlations with engagement_score:")
    print("-" * 60)
    
    top_corr = target_correlations.head(20)
    for feature, corr in top_corr.items():
        direction = "+" if corr > 0 else "-"
        print(f"  {feature:40} | {direction}{abs(corr):.4f}")
    
    # Visualize top correlations
    fig, ax = plt.subplots(figsize=(12, 8))
    
    colors = [COLORS['success'] if c > 0 else COLORS['danger'] for c in top_corr.values]
    bars = ax.barh(range(len(top_corr)), top_corr.values, color=colors)
    ax.set_yticks(range(len(top_corr)))
    ax.set_yticklabels(top_corr.index)
    ax.set_xlabel('Correlation with Engagement Score')
    ax.set_title('Top Feature Correlations with Target', fontsize=14, fontweight='bold')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.invert_yaxis()
    
    plt.tight_layout()
    save_figure(fig, 'target_correlations')
    plt.show()

In [ ]:
# Check for highly correlated feature pairs (multicollinearity)
print("\n" + "=" * 80)
print("⚠️ MULTICOLLINEARITY CHECK")
print("=" * 80)

# Find highly correlated pairs (|corr| > 0.8, excluding diagonal)
high_corr_threshold = 0.8
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)

print(f"\n📋 Highly Correlated Feature Pairs (|corr| > {high_corr_threshold}):")
print(f"   Found {len(high_corr_df)} pairs")
print("-" * 80)

if len(high_corr_df) > 0:
    display(high_corr_df)
    print("\n💡 Recommendation: Consider removing one feature from each highly correlated pair to reduce multicollinearity")
else:
    print("✅ No highly correlated feature pairs found!")

In [ ]:
# 5.6 Relationship to Target
print("\n" + "=" * 80)
print("📈 FEATURE-TARGET RELATIONSHIPS")
print("=" * 80)

# Select top features by correlation for scatter plots
if 'engagement_score' in df_engineered.columns:
    target_correlations = corr_matrix['engagement_score'].drop(['engagement_score',"like_count","view_count","comment_count","like_ratio","comment_ratio"]).sort_values(key=abs, ascending=False)
    top_features = target_correlations.head(6).index.tolist()
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, feature in enumerate(top_features):
        # Sample for performance
        sample = df_engineered[[feature, 'engagement_score']].dropna().sample(min(2000, len(df_engineered)), random_state=SEED)
        
        axes[i].scatter(sample[feature], sample['engagement_score'], alpha=0.3, 
                       color=COLORS['primary'], edgecolors='none', s=20)
        
        # Add trend line
        z = np.polyfit(sample[feature], sample['engagement_score'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(sample[feature].min(), sample[feature].max(), 100)
        axes[i].plot(x_line, p(x_line), color=COLORS['danger'], linewidth=2, linestyle='--')
        
        corr_val = target_correlations[feature]
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Engagement Score')
        axes[i].set_title(f'{feature}\n(r = {corr_val:.3f})', fontsize=10, fontweight='bold')
    
    plt.suptitle('Top Feature Relationships with Engagement Score', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, 'feature_target_scatter')
    plt.show()

In [ ]:
# Box plots for categorical features vs engagement_score
print("\n📊 Engagement Score by Categorical Features:")

categorical_features = ['subject', 'is_weekday', 'is_evening_upload', 'is_exam_focused', 'has_transcript']
existing_cat_features = [f for f in categorical_features if f in df_engineered.columns]

if 'engagement_score' in df_engineered.columns and len(existing_cat_features) > 0:
    fig, axes = plt.subplots(1, len(existing_cat_features), figsize=(5 * len(existing_cat_features), 5))
    if len(existing_cat_features) == 1:
        axes = [axes]
    
    for i, feature in enumerate(existing_cat_features):
        df_plot = df_engineered[[feature, 'engagement_score']].dropna()
        
        # Group by category
        categories = df_plot[feature].unique()
        data_by_category = [df_plot[df_plot[feature] == cat]['engagement_score'].values for cat in categories]
        
        bp = axes[i].boxplot(data_by_category, patch_artist=True)
        
        # Color boxes
        for patch in bp['boxes']:
            patch.set_facecolor(COLORS['primary'])
            patch.set_alpha(0.7)
        
        axes[i].set_xticklabels([str(c)[:15] for c in categories], rotation=45, ha='right')
        axes[i].set_ylabel('Engagement Score')
        axes[i].set_title(f'By {feature}', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, 'engagement_by_category')
    plt.show()

In [ ]:
# Part 5 Summary
print("\n" + "=" * 80)
print("⚙️ PART 5 SUMMARY: ENGINEERED FEATURES ANALYSIS")
print("=" * 80)

total_features = len(df_engineered.columns)
numerical_features = len(df_engineered.select_dtypes(include=[np.number]).columns)
missing_any = len(missing_data) if 'missing_data' in dir() else 0
high_corr_count = len(high_corr_df) if 'high_corr_df' in dir() else 0

# Calculate key insights from target analysis
target = df_engineered['engagement_score'].dropna() if 'engagement_score' in df_engineered.columns else None
skewness_val = f"{target.skew():.4f}" if target is not None else "N/A"

print( f""" 
📊 Key Findings:

1. DATASET OVERVIEW
   - Total features: {total_features}
   - Numerical features: {numerical_features}
   - Total samples: {len(df_engineered):,}

2. MISSING VALUES
   - Features with missing values: {missing_any}
   - Critical: Readability features have ~54% missing (linked to transcript availability)
   - Description has ~37% missing values
   
3. TARGET VARIABLE (engagement_score)
   - Skewness: {skewness_val} (right-skewed)
   - Consider log transformation for modeling
   - Outliers present in upper range

4. FEATURE QUALITY
   - Multiple highly skewed features identified
   - {high_corr_count} highly correlated feature pairs detected
   - Log transformation recommended for count-based features

5. KEY CORRELATIONS WITH TARGET
   - Strongest positive: view_count, like_count, comment_count (raw engagement metrics)
   - Strongest negative: (see correlation analysis above)

6. RECOMMENDATIONS FOR MODELING
   - Add 'has_transcript' indicator if not present
   - Fill transcript-related NaN with 0 where transcript not available
   - Apply log transformation to skewed count features
   - Consider removing one feature from multicollinear pairs
   - Use engagement_score as continuous target or engagement_category for classification

7. NEXT STEPS
   - Part 6: Cross-dataset analysis
   - Part 7: Data quality summary and final recommendations
""")

---
# Part 6: Cross-Dataset Analysis
Analyzing relationships between different data sources to understand patterns and data quality.

In [ ]:
# 6.1 Transcript × Video Join Analysis
print("=" * 80)
print("🔗 TRANSCRIPT × VIDEO JOIN ANALYSIS")
print("=" * 80)

# Merge videos with transcripts
video_transcript = df_raw[['video_id', 'title', 'view_count', 'like_count', 'comment_count', 'subject']].merge(
    df_transcripts[['video_id', 'transcript_available', 'transcript_language', 'is_generated']], 
    on='video_id', 
    how='left'
)

# Fill NaN transcript_available with False
video_transcript['transcript_available'] = video_transcript['transcript_available'].fillna(False)

# Calculate match rate
total_videos = len(video_transcript)
with_transcript = video_transcript['transcript_available'].sum()
without_transcript = total_videos - with_transcript

print(f"\n📊 Video-Transcript Match Summary:")
print("-" * 60)
print(f"  Total videos: {total_videos:,}")
print(f"  Videos with transcript: {with_transcript:,} ({with_transcript/total_videos*100:.1f}%)")
print(f"  Videos without transcript: {without_transcript:,} ({without_transcript/total_videos*100:.1f}%)")

In [ ]:
# Performance comparison: videos with vs without transcripts
print("\n" + "=" * 80)
print("📊 PERFORMANCE: WITH vs WITHOUT TRANSCRIPTS")
print("=" * 80)

# Group by transcript availability
perf_comparison = video_transcript.groupby('transcript_available').agg({
    'view_count': ['mean', 'median', 'std'],
    'like_count': ['mean', 'median'],
    'comment_count': ['mean', 'median']
}).round(2)

print("\n📋 Engagement Metrics Comparison:")
print("-" * 80)
display(perf_comparison)

# Statistical test (Mann-Whitney U test for non-normal distributions)
from scipy.stats import mannwhitneyu

with_trans_views = video_transcript[video_transcript['transcript_available'] == True]['view_count'].dropna()
without_trans_views = video_transcript[video_transcript['transcript_available'] == False]['view_count'].dropna()

if len(with_trans_views) > 0 and len(without_trans_views) > 0:
    stat, p_value = mannwhitneyu(with_trans_views, without_trans_views, alternative='two-sided')
    print(f"\n📈 Mann-Whitney U Test (View Count):")
    print(f"   U-statistic: {stat:,.0f}")
    print(f"   p-value: {p_value:.4e}")
    print(f"   Significant difference: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['view_count', 'like_count', 'comment_count']
titles = ['View Count', 'Like Count', 'Comment Count']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    data_with = video_transcript[video_transcript['transcript_available'] == True][metric].dropna()
    data_without = video_transcript[video_transcript['transcript_available'] == False][metric].dropna()
    
    # Use log scale for better visualization
    bp = axes[i].boxplot([np.log1p(data_with), np.log1p(data_without)], 
                          patch_artist=True, labels=['With Transcript', 'Without Transcript'])
    
    colors_box = [COLORS['success'], COLORS['warning']]
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[i].set_ylabel(f'Log({title} + 1)')
    axes[i].set_title(f'{title} Distribution', fontsize=12, fontweight='bold')

plt.suptitle('Performance Comparison: With vs Without Transcripts', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_figure(fig, 'performance_with_vs_without_transcript')
plt.show()

In [ ]:
# Subject distribution for transcript-available videos
print("\n" + "=" * 80)
print("📚 TRANSCRIPT AVAILABILITY BY SUBJECT")
print("=" * 80)

if 'subject' in video_transcript.columns:
    subject_transcript = video_transcript.groupby('subject').agg(
        total_videos=('video_id', 'count'),
        with_transcript=('transcript_available', 'sum')
    ).reset_index()
    subject_transcript['coverage_pct'] = (subject_transcript['with_transcript'] / 
                                          subject_transcript['total_videos'] * 100).round(1)
    subject_transcript = subject_transcript.sort_values('total_videos', ascending=False)
    
    print("\n📋 Coverage by Subject:")
    display(subject_transcript)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(subject_transcript))
    width = 0.35
    
    bars1 = ax.bar([i - width/2 for i in x], subject_transcript['total_videos'], width, 
                   label='Total Videos', color=COLORS['primary'], alpha=0.7)
    bars2 = ax.bar([i + width/2 for i in x], subject_transcript['with_transcript'], width, 
                   label='With Transcript', color=COLORS['success'], alpha=0.7)
    
    ax.set_xlabel('Subject')
    ax.set_ylabel('Number of Videos')
    ax.set_title('Transcript Coverage by Subject', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(subject_transcript['subject'], rotation=45, ha='right')
    ax.legend()
    
    # Add percentage labels
    for i, (idx, row) in enumerate(subject_transcript.iterrows()):
        ax.annotate(f'{row["coverage_pct"]:.0f}%',
                   xy=(i, max(row['total_videos'], row['with_transcript']) + 100),
                   ha='center', fontsize=9, fontweight='bold', color=COLORS['info'])
    
    plt.tight_layout()
    save_figure(fig, 'transcript_by_subject')
    plt.show()

In [ ]:
# 6.2 Channel × Video Analysis
print("\n" + "=" * 80)
print("📺 CHANNEL × VIDEO ANALYSIS")
print("=" * 80)

# Merge channel data with video performance
channel_performance = df_raw.groupby('channel_id').agg(
    video_count=('video_id', 'count'),
    avg_views=('view_count', 'mean'),
    median_views=('view_count', 'median'),
    total_views=('view_count', 'sum'),
    avg_likes=('like_count', 'mean'),
    avg_comments=('comment_count', 'mean')
).reset_index()

# Merge with channel names
if 'channel_id' in df_channels.columns and 'title' in df_channels.columns:
    channel_performance = channel_performance.merge(
        df_channels[['channel_id', 'title', 'subscriber_count']], 
        on='channel_id', 
        how='left'
    )

channel_performance = channel_performance.sort_values('total_views', ascending=False)

print("\n📋 Top 10 Channels by Total Views:")
display(channel_performance.head(10))

In [ ]:
# Visualize channel performance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top 10 channels by total views
top_channels = channel_performance.head(10)

# Total views bar chart
axes[0].barh(range(len(top_channels)), top_channels['total_views'], color=COLORS['primary'])
axes[0].set_yticks(range(len(top_channels)))
if 'title' in top_channels.columns:
    axes[0].set_yticklabels([str(t)[:25] for t in top_channels['title']])
else:
    axes[0].set_yticklabels([str(c)[:15] for c in top_channels['channel_id']])
axes[0].set_xlabel('Total Views')
axes[0].set_title('Top 10 Channels by Total Views', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# Scatter: Video count vs Avg views
axes[1].scatter(channel_performance['video_count'], 
                channel_performance['avg_views'], 
                c=COLORS['primary'], alpha=0.6, s=80, edgecolors='white')
axes[1].set_xlabel('Number of Videos')
axes[1].set_ylabel('Average Views per Video')
axes[1].set_title('Channel Video Count vs Average Views', fontsize=12, fontweight='bold')

# Add trend line
z = np.polyfit(channel_performance['video_count'], channel_performance['avg_views'], 1)
p = np.poly1d(z)
x_line = np.linspace(channel_performance['video_count'].min(), channel_performance['video_count'].max(), 100)
axes[1].plot(x_line, p(x_line), color=COLORS['danger'], linewidth=2, linestyle='--', label='Trend')
axes[1].legend()

plt.tight_layout()
save_figure(fig, 'channel_performance')
plt.show()

In [ ]:
# Top vs Bottom performing channels
print("\n" + "=" * 80)
print("🏆 TOP vs BOTTOM PERFORMING CHANNELS")
print("=" * 80)

# Sort by average views
sorted_by_avg = channel_performance.sort_values('avg_views', ascending=False)

print("\n🥇 Top 5 Channels by Average Views:")
print("-" * 80)
top_5 = sorted_by_avg.head(5)
for i, (idx, row) in enumerate(top_5.iterrows(), 1):
    name = row.get('title', row['channel_id'])[:30]
    print(f"  {i}. {name}: {row['avg_views']:,.0f} avg views, {row['video_count']} videos")

print("\n🥉 Bottom 5 Channels by Average Views:")
print("-" * 80)
bottom_5 = sorted_by_avg.tail(5)
for i, (idx, row) in enumerate(bottom_5.iterrows(), 1):
    name = row.get('title', row['channel_id'])[:30]
    print(f"  {i}. {name}: {row['avg_views']:,.0f} avg views, {row['video_count']} videos")

In [ ]:
# Subject distribution across channels
print("\n" + "=" * 80)
print("📚 SUBJECT DISTRIBUTION ACROSS CHANNELS")
print("=" * 80)

if 'subject' in df_raw.columns:
    channel_subject = df_raw.groupby(['channel_id', 'subject']).size().unstack(fill_value=0)
    
    # Get top channels
    top_channel_ids = channel_performance.head(10)['channel_id'].values
    channel_subject_top = channel_subject.loc[channel_subject.index.isin(top_channel_ids)]
    
    # Visualize
    fig, ax = plt.subplots(figsize=(14, 8))
    
    channel_subject_top.plot(kind='barh', stacked=True, ax=ax, colormap='tab10')
    ax.set_xlabel('Number of Videos')
    ax.set_ylabel('Channel')
    ax.set_title('Subject Distribution in Top 10 Channels', fontsize=14, fontweight='bold')
    ax.legend(title='Subject', bbox_to_anchor=(1.02, 1), loc='upper left')
    
    plt.tight_layout()
    save_figure(fig, 'subject_by_channel')
    plt.show()

In [ ]:
# Part 6 Summary
print("\n" + "=" * 80)
print("🔗 PART 6 SUMMARY: CROSS-DATASET ANALYSIS")
print("=" * 80)

print(f"""
📊 Key Cross-Dataset Findings:

1. TRANSCRIPT × VIDEO ANALYSIS
   - {with_transcript:,} videos ({with_transcript/total_videos*100:.1f}%) have transcripts
   - Performance difference detected between videos with/without transcripts
   - Transcript coverage varies significantly by subject

2. CHANNEL × VIDEO ANALYSIS
   - {len(channel_performance)} channels in dataset
   - Large variation in channel performance (views, engagement)
   - Some channels specialize in specific subjects

3. DATA INTEGRATION INSIGHTS
   - All datasets can be joined via video_id and channel_id
   - No orphan records detected (all videos have corresponding channel data)
   - Transcript data covers majority of videos

4. RECOMMENDATIONS
   - Use 'has_transcript' as a feature in modeling
   - Consider channel-level features for model enhancement
   - Subject may be a strong predictor - include in feature set
""")

---
# Part 7: Data Quality Summary & Recommendations
Final summary of data quality issues and actionable recommendations for model training.

In [ ]:
# 7.1 Data Quality Report
print("=" * 80)
print("📋 COMPREHENSIVE DATA QUALITY REPORT")
print("=" * 80)

quality_issues = []

# 1. Check Missing Description
# Use df_raw as it's the source of truth for descriptions
desc_missing = df_raw['description'].isna().sum() if 'description' in df_raw.columns else 0
desc_pct = (desc_missing / len(df_raw) * 100)
quality_issues.append({
    'Issue': 'Missing description',
    'Severity': 'Medium',
    'Affected_Rows': desc_missing,
    'Percentage': f'{desc_pct:.1f}%',
    'Recommendation': 'Fill with empty string'
})

# 2. Check Missing Transcripts
# Calculate from df_transcripts if available, or try to infer
total_videos = len(df_raw)
if 'transcript_available' in df_transcripts.columns:
    transcript_count = df_transcripts['transcript_available'].sum()
else:
    # Fallback: count non-empty transcript text
    transcript_count = df_transcripts['transcript_text'].notna().sum()

transcripts_missing = total_videos - transcript_count
transcript_pct = (transcripts_missing / total_videos * 100)

quality_issues.append({
    'Issue': 'Missing transcripts',
    'Severity': 'High',
    'Affected_Rows': transcripts_missing,
    'Percentage': f'{transcript_pct:.1f}%',
    'Recommendation': 'Add has_transcript flag, fill derived features with 0'
})

# 3. Check Engineered Features (Readability)
# These are often NaN when transcripts are missing
if 'flesch_reading_ease' in df_engineered.columns:
    readability_missing = df_engineered['flesch_reading_ease'].isna().sum()
    readability_pct = (readability_missing / len(df_engineered) * 100)
    
    severity = 'High' if readability_pct > 20 else 'Medium'
    quality_issues.append({
        'Issue': 'Missing readability features',
        'Severity': severity,
        'Affected_Rows': readability_missing,
        'Percentage': f'{readability_pct:.1f}%',
        'Recommendation': 'Fill with 0 (as these likely map to no transcript)'
    })

# 4. Check Skewed Features
# Identify highly skewed numerical features impacting linear models
numerical_cols = df_engineered.select_dtypes(include=[np.number]).columns
skewed_cols = [col for col in numerical_cols if abs(df_engineered[col].skew()) > 2]
if skewed_cols:
    quality_issues.append({
        'Issue': 'Highly skewed features (>2 skew)',
        'Severity': 'Medium',
        'Affected_Rows': f"{len(skewed_cols)} features",
        'Percentage': 'N/A',
        'Recommendation': 'Apply Log1p transformation'
    })

# Display the report
quality_df = pd.DataFrame(quality_issues)
print("\n📋 Data Quality Issues Detected:")
print("-" * 100)
# Reorder columns for readability
cols = ['Issue', 'Severity', 'Percentage', 'Affected_Rows', 'Recommendation']
display(quality_df[cols])

In [ ]:
# 7.2 Recommended Cleaning Steps
print("\n" + "=" * 80)
print("🔧 RECOMMENDED CLEANING STEPS (Copy & Paste for Model Training)")
print("=" * 80)

print(f"""
def clean_data_for_modeling(df):
    
    # 1. Handle Missing Descriptions
    if 'description' in df.columns:
        df['description'] = df['description'].fillna('')
        
    # 2. Add Transcript Availability Flag
    # Assuming 'transcript_word_count' > 0 implies a transcript exists
    if 'transcript_word_count' in df.columns:
        df['has_transcript'] = df['transcript_word_count'].fillna(0) > 0
        df['has_transcript'] = df['has_transcript'].astype(int)

    # 3. Operations for Transcript-Derived Features (Fill with 0)
    # Rationale: If no transcript, count-based metrics are 0.
    transcript_cols = [
        'transcript_word_count', 'transcript_char_count', 'transcript_sentence_count',
        'question_count', 'example_count', 'explanation_count', 'contrast_count', 'technical_term_count',
        'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog_index', 'automated_readability_index'
    ]
    
    for col in transcript_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 4. Log Transformation for Highly Skewed Features
    # Skewed features identified: {skewed_cols if 'skewed_cols' in locals() else 'view_count, like_count...'}
    skewed_features = ['view_count', 'like_count', 'comment_count', 'days_since_publish', 'channel_video_count']
    
    for col in skewed_features:
        if col in df.columns:
            # Create log features, handling zeros with log1p
            df[f'log_{{col}}'] = np.log1p(df[col])
            
    return df
""")

In [ ]:
# 7.3 Feature Selection Insights
print("\n" + "=" * 80)
print("🎯 FEATURE SELECTION INSIGHTS")
print("=" * 80)

# Identify Baseline Features
baseline_features = [
    # Temporal
    'days_since_publish', 'is_weekday', 'publish_hour_sin', 'publish_hour_cos',
    
    # Engagement (Prior)
    'channel_avg_views', 'channel_subscriber_count',
    
    # Content (Metadata)
    'title_length', 'is_exam_focused', 'tag_count',
    
    # Content (Transcript - High Value)
    'has_transcript', 'transcript_word_count',
    
    # Ratios (often better predictors than raw counts)
    'like_ratio', 'comment_ratio'
]

print(f"\n✅ Recommended Baseline Features ({len(baseline_features)}):")
print("-" * 60)
for i, feat in enumerate(sorted(baseline_features)):
    status = "Available" if feat in df_engineered.columns else "Needs Creation"
    if feat == 'has_transcript': status = "Needs Creation (from cleaning step)"
    print(f"{i+1:02d}. {feat:<30} [{status}]")

print("\n⚠️ Features to Exclude/Drop:")
print("-" * 60)
print("1. Raw ID columns: video_id, channel_id")
print("2. Raw text columns: title, description, transcript_text")
print("3. Leaky targets (if strictly predicting future): view_count (use channel priors instead)")

In [ ]:
# Final Notebook Summary
print("\n" + "=" * 80)
print("🎉 EDA NOTEBOOK COMPLETE - FINAL SUMMARY")
print("=" * 80)

print(f"""
📊 DATASET STATUS:
   - Raw Videos:    {len(df_raw):,}
   - Transcripts:   {len(df_transcripts):,}
   - Match Rate:    {transcript_pct:.1f}% of videos are missing transcripts.

🔍 KEY FINDINGS:
   1. The 'Missing Readability' issue ({readability_pct:.1f}%) is directly caused by missing transcripts.
   2. 'Description' is missing for {desc_pct:.1f}% of videos, but titles are 100% present.
   3. Engagement metrics are highly right-skewed (log-transform strongly recommended).

🚀 NEXT STEPS:
   1. Implement the `clean_data_for_modeling` function defined above.
   2. Regenerate `videos_engineered.csv` using the cleaning logic.
   3. Proceed to `02_model_baseline_testing.ipynb` with the clean dataset.
""")

print(f"\n✅ Analysis completed at: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

---

## Part 5: Engagement Analysis

Understanding what drives engagement. We examine correlations between metrics and identify top-performing content.

In [ ]:
# Metrics & Correlation
print("\n" + "="*80)
print("📊 ENGAGEMENT ANALYSIS")
print("="*80)

# Calculate Engagement Rate (Likes + Comments / Views)
# Handle division by zero implicitly (pandas returns inf/nan, we can clean if needed)
df_videos['engagement_rate'] = (df_videos['like_count'] + df_videos['comment_count']) / df_videos['view_count'] * 100

# Descriptive Stats
print("\n📈 Descriptive Statistics (Engagement):")
metrics = ['view_count', 'like_count', 'comment_count', 'engagement_rate', 'duration_seconds']
print(df_videos[metrics].describe())

# Correlation Matrix
corr = df_videos[metrics].corr()
print("\n🔗 Correlation Matrix:")
print(corr)

# Heatmap Visualization
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax)
ticks = range(len(metrics))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(metrics, rotation=45, ha='left')
ax.set_yticklabels(metrics)
ax.set_title("Correlation Matrix of Video Metrics", pad=20)

for (i, j), z in np.ndenumerate(corr):
    ax.text(j, i, '{:0.2f}'.format(z), ha='center', va='center',
            color='white' if abs(z) > 0.5 else 'black')

plt.tight_layout()
save_figure(fig, 'engagement_correlation')
plt.show()


In [ ]:
# Top Videos Analysis
print("\n🏆 Top 10 Videos by Views:")
top_views = df_videos.nlargest(10, 'view_count')[['title', 'channel_title', 'view_count', 'engagement_rate']]
print(top_views.to_string(index=False))

print("\n💎 Top 10 Videos by Engagement Rate (min 1000 views):")
# Filter for significant views to avoid outliers with 1 view and 1 like
top_engagement = df_videos[df_videos['view_count'] > 1000].nlargest(10, 'engagement_rate')[['title', 'channel_title', 'view_count', 'engagement_rate']]
print(top_engagement.to_string(index=False))

# Visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

# Plot Top Views
y_pos = range(len(top_views))
ax1.barh(y_pos, top_views['view_count'], color=COLORS['primary'])
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_views['title'].str[:40] + '...')
ax1.set_xlabel('Views')
ax1.set_title('Top 10 Videos by Views')
ax1.invert_yaxis()

# Plot Top Engagement
y_pos2 = range(len(top_engagement))
ax2.barh(y_pos2, top_engagement['engagement_rate'], color=COLORS['success'])
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(top_engagement['title'].str[:40] + '...')
ax2.set_xlabel('Engagement Rate (%)')
ax2.set_title('Top 10 Videos by Engagement Rate (>1k Views)')
ax2.invert_yaxis()

plt.tight_layout()
save_figure(fig, 'top_videos_analysis')
plt.show()


---

## Part 6: Conclusion and Next Steps

### 🔎 Key Findings

1.  **Video Metadata**:
    - We have analyzed the video metadata and identified the distribution of duration, views, and engagement metrics.
    - Missing values have been identified and visualized.

2.  **Transcripts**:
    - A significant portion of videos have transcripts available, primarily in English.
    - The main reasons for missing transcripts have been categorized (e.g., Bot Detection, Video Unavailable).

3.  **Engagement**:
    - Engagement rate varies significantly across videos.
    - We observed correlations between view counts and engagement metrics.

### 🚀 Next Steps

This exploration sets the foundation for more advanced analysis. Future work can focus on:

-   **Sentiment Analysis**: Determining the sentiment of the video transcripts (Positive/Negative/Neutral).
-   **Topic Modeling**: Using LDA or BERTopic to discover hidden topics within the transcripts.
-   **Named Entity Recognition (NER)**: Extracting key entities (people, organizations, locations) mentioned in the videos.
-   **Predictive Modeling**: Building models to predict video performance based on metadata and transcript content.
